---
title: "Chương 4: Kết quả thu thập"
short_title: "4. Kết quả thu thập"
---

Chương này trình bày các số liệu mà đề bài yêu cầu: số bài crawl thành công, tỷ lệ thành công, kích thước trung bình của một bài trước và sau khi bỏ hình, số reference trung bình mỗi bài, và tỷ lệ lấy được metadata của reference. Mọi số liệu được tính lại từ trạng thái cuối cùng lưu trên đĩa của từng thư mục dữ liệu, với mẫu số được ghi rõ.

**Định nghĩa dùng trong chương:**

- **Bài crawl thành công:** mọi phiên bản đã ở trạng thái cuối (`success`, `pdf_only` hoặc `no_tex`, có thư mục trên đĩa và danh sách file khớp), có `metadata.json`, và references ở trạng thái `success` hoặc `empty`. Đây là chỉ số `papers_pipeline_core_complete` của `corpus_statistics.py`.
- **Kích thước trước khi bỏ hình:** tổng dung lượng **chưa nén** của mọi file trong gói nguồn, cộng trên các phiên bản của bài. **Sau khi bỏ hình:** tổng dung lượng các file `.tex`/`.bib` được giữ. Hai số dùng cùng một cơ sở đo. Chỉ tính các bài có mọi phiên bản ở trạng thái cuối.
- **Số reference trung bình:** trung bình trên các bài có references ở trạng thái `success` hoặc `empty`, trong đó bài `empty` được tính là 0 reference. Bài lỗi API không bị tính thành 0.

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils import (COLORS, FINAL_SOURCE, REF_OK, group, human_bytes, human_seconds,
                          load, members, meta, metric, mib, pct, provenance, show, table)
provenance()

In [ ]:
papers = load("papers")
versions = load("versions")
m, g = members(), group()

found = papers[papers["discovery_status"] == "success"]
# assigned IDs: the official lists when exported with --assigned-ids, else the IDs of the manifests
assigned = bool(meta().get("assigned_ids"))
ids_total = int(metric("assigned_ids_expected")) if assigned else len(papers)
ids_missing = int(metric("assigned_ids_missing_from_manifest")) if assigned else None
core = metric("papers_pipeline_core_complete")
source_complete = metric("papers_source_complete")
strict = metric("papers_pipeline_complete_strict")

# sizes per paper, over papers whose every version reached a final state
final = versions["status"].isin(FINAL_SOURCE)
complete_ids = versions.assign(final=final).groupby("arxiv_id")["final"].all()
complete_ids = complete_ids[complete_ids].index
sizes = (versions[versions["arxiv_id"].isin(complete_ids)]
         .groupby(["member", "arxiv_id"])[["downloaded_bytes", "original_bytes", "image_bytes", "kept_bytes"]]
         .sum(min_count=1).fillna(0).reset_index())
with_source = sizes[sizes["original_bytes"] > 0]

refs = found[found["ref_status"].isin(REF_OK)]
n_refs = refs["n_references"].sum()
n_arxiv = refs["n_with_arxiv_id"].sum()

rows = [
    ("Số ID được giao" if assigned else "Số ID được giao (theo manifest)", f"{ids_total:,}", "ID"),
    ("Số ID được giao nhưng không có trong dữ liệu", f"{ids_missing:,}" if assigned else "không kiểm tra", "ID"),
    ("Số bài tồn tại trên arXiv", f"{len(found):,}", "bài"),
    ("Số bài crawl thành công", f"{core:,.0f}", "bài"),
    ("Tỷ lệ crawl thành công / bài tồn tại", pct(core / len(found)), ""),
    ("Tỷ lệ crawl thành công / ID được giao", pct(core / ids_total), ""),
    ("Số bài có đủ mã nguồn mọi phiên bản", f"{source_complete:,.0f}", "bài"),
    ("Số bài thành công và đã có đủ ngày nộp arXiv của reference", f"{strict:,.0f}", "bài"),
    ("Độ phủ phiên bản (trạng thái cuối / phiên bản cần tải)",
     pct(metric("versions_final_record_and_folder") / metric("versions_expected")), ""),
    ("Kích thước TB mỗi bài trước khi bỏ hình (chưa nén)", human_bytes(with_source["original_bytes"].mean()), "/bài"),
    ("Kích thước TB mỗi bài sau khi bỏ hình (.tex/.bib)", human_bytes(with_source["kept_bytes"].mean()), "/bài"),
    ("Dung lượng giảm (1 − tổng giữ lại / tổng gốc)",
     pct(1 - with_source["kept_bytes"].sum() / with_source["original_bytes"].sum()), ""),
    ("Dung lượng TB file nén tải về mỗi bài", human_bytes(with_source["downloaded_bytes"].mean()), "/bài"),
    ("Số reference trung bình mỗi bài", f"{n_refs / len(refs):,.2f}", f"reference/bài, trên {len(refs):,} bài"),
    ("Tỷ lệ bài lấy được references (success hoặc empty) / bài tồn tại", pct(len(refs) / len(found)), ""),
    ("Tỷ lệ reference có metadata đầy đủ (complete)", pct(refs["n_complete"].sum() / n_refs), ""),
    ("Tỷ lệ reference có arXiv ID", pct(n_arxiv / n_refs), ""),
    ("Tỷ lệ reference có arXiv ID đã có ngày nộp arXiv", pct(refs["n_with_ref_date"].sum() / n_arxiv), ""),
]
table(pd.DataFrame(rows, columns=["Chỉ số", "Giá trị", "Đơn vị / mẫu số"]),
      "Bảng 4.1. Các chỉ số chính của cả nhóm")

## 1. Theo từng thành viên

In [ ]:
def per_member(name, df):
    f = df[df["discovery_status"] == "success"]
    r = f[f["ref_status"].isin(REF_OK)]
    s = with_source[with_source["member"] == name]
    stats = load("statistics")
    core_m = pd.to_numeric(stats[(stats["member"] == name) & (stats["metric"] == "papers_pipeline_core_complete")]["value"]).sum()
    return {"Thành viên": name, "ID": len(df), "Bài tồn tại": len(f), "Crawl thành công": int(core_m),
            "Tỷ lệ thành công": pct(core_m / len(f)) if len(f) else "–",
            "TB trước khi bỏ hình": human_bytes(s["original_bytes"].mean()),
            "TB sau khi bỏ hình": human_bytes(s["kept_bytes"].mean()),
            "Reference TB/bài": f"{r['n_references'].mean():.2f}" if len(r) else "–",
            "Bài có references": pct(len(r) / len(f)) if len(f) else "–"}

table(pd.DataFrame([per_member(n, d) for n, d in papers.groupby("member")]),
      "Bảng 4.2. Các chỉ số chính theo thành viên")

## 2. Trạng thái các phiên bản

Mọi phiên bản trong danh sách của discovery đều có một trạng thái; phiên bản không có TeX được ghi nhận chứ không bị bỏ.

In [ ]:
st = versions.groupby(["member", "status"]).size().reset_index(name="versions")
fig = px.bar(st, x="member", y="versions", color="status", color_discrete_sequence=COLORS,
             title="Trạng thái cuối của các phiên bản theo thành viên",
             labels={"member": "Thành viên", "versions": "Số phiên bản", "status": "Trạng thái"})
show(fig)
total = versions["status"].value_counts().rename_axis("Trạng thái").reset_index(name="Số phiên bản")
total["Tỷ lệ"] = (total["Số phiên bản"] / total["Số phiên bản"].sum()).map(pct)
table(total, "Bảng 4.3. Trạng thái cuối của các phiên bản, cả nhóm")

## 3. Kích thước bài báo trước và sau khi bỏ hình

Trục tung dùng thang log vì kích thước trải rộng nhiều bậc độ lớn. Chỉ tính các bài có mã nguồn TeX (dung lượng gốc lớn hơn 0).

In [ ]:
long = with_source.melt(id_vars=["member", "arxiv_id"], value_vars=["original_bytes", "kept_bytes"],
                        var_name="kind", value_name="bytes")
long["kind"] = long["kind"].map({"original_bytes": "Trước (mọi file, chưa nén)", "kept_bytes": "Sau (.tex/.bib)"})
long = long[long["bytes"] > 0]
long["MB"] = long["bytes"] / 2**20
fig = px.box(long, x="kind", y="MB", color="kind", log_y=True, color_discrete_sequence=COLORS,
             title="Phân bố kích thước mỗi bài trước và sau khi bỏ hình",
             labels={"MB": "Kích thước mỗi bài (MB, thang log)", "kind": ""})
fig.update_layout(showlegend=False)
show(fig)
ratio = 1 - with_source["kept_bytes"] / with_source["original_bytes"]
fig = px.histogram(x=ratio, nbins=50, color_discrete_sequence=COLORS,
                   title="Tỷ lệ dung lượng bỏ đi của từng bài (1 − giữ lại / gốc)")
fig.update_layout(showlegend=False, xaxis_title="Tỷ lệ bỏ đi", yaxis_title="Số bài")
fig.update_xaxes(tickformat=".0%")
show(fig)
desc = pd.DataFrame({
    "Thống kê": ["Trung bình", "Trung vị", "Phân vị 90", "Lớn nhất"],
    "Trước khi bỏ hình": [human_bytes(x) for x in with_source["original_bytes"].agg(["mean", "median"]).tolist()
                          + [with_source["original_bytes"].quantile(0.9), with_source["original_bytes"].max()]],
    "Sau khi bỏ hình": [human_bytes(x) for x in with_source["kept_bytes"].agg(["mean", "median"]).tolist()
                        + [with_source["kept_bytes"].quantile(0.9), with_source["kept_bytes"].max()]]})
table(desc, f"Bảng 4.4. Kích thước mỗi bài, {len(with_source):,} bài có mã nguồn TeX")
print(f"Riêng file hình chiếm {pct(with_source['image_bytes'].sum() / with_source['original_bytes'].sum())} "
      "tổng dung lượng gốc; phần còn lại bị bỏ là các file khác ngoài .tex/.bib (class, style, .bbl, dữ liệu...).")

## 4. References

Trạng thái lấy references của mỗi bài (`missing` là bài chưa có file references, thường do API lỗi ở lần chạy cuối):

In [ ]:
rs = found["ref_status"].fillna("missing").value_counts().rename_axis("status").reset_index(name="papers")
fig = px.bar(rs, x="status", y="papers", text="papers", color_discrete_sequence=COLORS,
             title=f"Trạng thái references của {len(found):,} bài tồn tại",
             labels={"status": "Trạng thái", "papers": "Số bài"})
show(fig)
fig = px.histogram(refs, x="n_references", nbins=80, color_discrete_sequence=COLORS,
                   title=f"Số reference mỗi bài ({len(refs):,} bài có references success/empty)",
                   labels={"n_references": "Số reference"})
fig.update_layout(yaxis_title="Số bài")
show(fig)
q = refs.groupby("member")[["n_complete", "n_incomplete", "n_unresolved"]].sum().reset_index()
q = q.melt(id_vars="member", var_name="metadata_status", value_name="references")
q["metadata_status"] = q["metadata_status"].str.removeprefix("n_")
fig = px.bar(q, x="member", y="references", color="metadata_status", color_discrete_sequence=COLORS,
             title="Chất lượng metadata của reference theo thành viên",
             labels={"member": "Thành viên", "references": "Số reference", "metadata_status": "metadata_status"})
show(fig)